# 3901161_1386_EasyTSLite.csv

This CSV is an Argo profile file with a metadata comment block at the top and tabular measurements below.

## File Structure

The first lines begin with `#`. These are comments used as metadata, not data rows. They describe the profile and the source of the file.

In this file, the metadata includes:

- format: `EasyOneArgoTSLite`
- creation date: `2026-03-16T04:30:37Z`
- creation centre: `Ifremer`
- data source DOI: `https://doi.org/10.17882/42182#126945`
- platform number: `3901161`
- cycle number: `1386`
- profile date: `2017-02-09T23:29:42Z`
- profile latitude: `6.144`
- profile longitude: `-119.519`

The comment lines also define the variables used in the profile:

- pressure = sea water pressure, measured in decibar
- temperature = sea temperature in-situ on the ITS-90 scale
- salinity = practical salinity

After the comments, the file switches to a comma-separated header row:

- pressure (decibar)
- temperature (degree_celsius)
- salinity (dimensionless)
- pressure_error (decibar)
- temperature_error (degree_celsius)
- salinity_error (dimensionless)

Each following row is one measurement at a given pressure level.

## How To Read It In Pandas

Because the metadata lines start with `#`, pandas should skip them when reading the file:

```python
single_argo = pd.read_csv('3901161_1386_EasyTSLite.csv', comment='#')
```

That keeps the header row and measurement rows while ignoring the comment block.

## Notes

- The parser error happens if pandas tries to interpret the comment block as tabular data.
- The file is comma-separated, so the default delimiter is correct once the comment lines are skipped.

In [24]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re


csv_path = Path('3901161_1386_EasyTSLite.csv')
argo_metadata = {}
with csv_path.open('r', encoding='utf-8', errors='replace') as csv_file:
    for line in csv_file:
        if not line.startswith('#'):
            break
        comment = line[1:].strip()
        if '=' in comment:
            key, value = comment.split('=', 1)
            argo_metadata[key.strip()] = value.strip()
        else:
            parts = comment.split(None, 1)
            if len(parts) == 2:
                argo_metadata[parts[0].strip()] = parts[1].strip()
            else:
                argo_metadata[comment] = None

single_argo = pd.read_csv(csv_path, comment='#')

# Add one constant metadata column per key for every row in the profile table.
for key, value in argo_metadata.items():
    safe_key = re.sub(r'[^0-9a-zA-Z_]+', '_', key.strip().lower()).strip('_')
    meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
    single_argo[meta_col] = value

single_argo.attrs['metadata'] = argo_metadata
single_argo.head()

,pressure (decibar),temperature (degree_celsius),salinity (dimensionless),pressure_error (decibar),temperature_error (degree_celsius),salinity_error (dimensionless),meta_format,meta_creation_date,meta_creation_centre,meta_creation_centre_pid,...,meta_platform_number,meta_cycle_number,meta_direction_of_profile,meta_data_mode,meta_profile_date,meta_profile_latitude,meta_profile_longitude,meta_pressure,meta_temperature,meta_salinity
0,2.0,27.643,34.076,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
1,5.0,27.636,34.077,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
2,10.0,27.528,34.076,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
3,15.0,27.435,34.071,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
4,20.0,27.424,34.071,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity


In [25]:
from pathlib import Path
import pandas as pd
import re


def _parse_argo_metadata(csv_file):
    metadata = {}
    with Path(csv_file).open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.startswith("#"):
                break
            comment = line[1:].strip()
            if not comment:
                continue
            if "=" in comment:
                key, value = comment.split("=", 1)
                metadata[key.strip()] = value.strip()
            else:
                parts = comment.split(None, 1)
                if len(parts) == 2:
                    metadata[parts[0].strip()] = parts[1].strip()
                else:
                    metadata[comment] = None
    return metadata


def combine_csvs_from_subfolders(
    root_folder,
    pattern="*.csv",
    recursive=True,
    read_kwargs=None,
    max_files=None,
    include_metadata=True,
):
    """
    Read CSV files from a folder (optionally including subfolders)
    and combine them into a single pandas DataFrame.

    Parameters
    ----------
    root_folder : str | Path
        Folder that contains CSV files directly or inside subfolders.
    pattern : str, default "*.csv"
        File name pattern to match.
    recursive : bool, default True
        If True, search all subfolders with rglob; otherwise only top-level files.
    read_kwargs : dict | None
        Optional keyword args passed to pd.read_csv (e.g., {"comment": "#"}).
    max_files : int | None
        Optional limit on number of files to read (useful for large folders).
    include_metadata : bool, default True
        If True, parse comment-block metadata and append as columns.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame containing all matched CSV files.
    """
    root = Path(root_folder)
    if not root.exists() or not root.is_dir():
        raise ValueError(f"Invalid folder path: {root}")

    read_kwargs = read_kwargs or {}
    csv_files = sorted(root.rglob(pattern) if recursive else root.glob(pattern))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under: {root}")

    if max_files is not None:
        if max_files <= 0:
            raise ValueError("max_files must be a positive integer")
        csv_files = csv_files[:max_files]

    frames = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, **read_kwargs)

        if include_metadata:
            metadata = _parse_argo_metadata(csv_file)
            for key, value in metadata.items():
                safe_key = re.sub(r"[^0-9a-zA-Z_]+", "_", key.strip().lower()).strip("_")
                meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
                df[meta_col] = value

        frames.append(df)

    return pd.concat(frames, ignore_index=True)



In [34]:
# Find the top Southern California platform from the index and load it.
index_path = Path("EasyOneArgoTSLite_20260316T043037Z/EasyOneArgoTSLite_index.csv")
index_df = pd.read_csv(index_path, skiprows=5)
sc_mask = (
    index_df["profile_latitude"].between(32, 34.8)
    & index_df["profile_longitude"].between(-125, -117)
)
sc_index = index_df.loc[sc_mask].copy()
platform_counts = (
    sc_index.groupby("platform_number")
    .size()
    .sort_values(ascending=False)
)
top_10_socal_platforms = platform_counts.head(10)
best_socal_platform = top_10_socal_platforms.index[0]
platform_folder = Path("EasyOneArgoTSLite_20260316T043037Z/data") / str(best_socal_platform)

single_argo = combine_csvs_from_subfolders(
    platform_folder,
    read_kwargs={"comment": "#"},
    include_metadata=True,
)

single_argo.shape

meta_cols = [c for c in single_argo.columns if c.startswith("meta_")]
print("Southern California coastal rows:", len(sc_index))
print("Top 10 platform numbers by profile count in the region:")
print(top_10_socal_platforms.to_string())
print()
print("Top platform:", best_socal_platform)
print("Rows:", len(single_argo))
print("Metadata columns:", len(meta_cols))
print("First metadata columns:", meta_cols[:8])
single_argo.head()

Southern California coastal rows: 2157
Top 10 platform numbers by profile count in the region:
platform_number
4901639    161
5904318    124
4903296    106
2903886    100
4900848     93
4901640     88
4901408     87
5906445     80
4900626     76
5906294     70

Top platform: 4901639
Rows: 17816
Metadata columns: 16
First metadata columns: ['meta_format', 'meta_creation_date', 'meta_creation_centre', 'meta_creation_centre_pid', 'meta_data_source_doi', 'meta_data_centre', 'meta_platform_number', 'meta_cycle_number']


,pressure (decibar),temperature (degree_celsius),salinity (dimensionless),pressure_error (decibar),temperature_error (degree_celsius),salinity_error (dimensionless),meta_format,meta_creation_date,meta_creation_centre,meta_creation_centre_pid,...,meta_platform_number,meta_cycle_number,meta_direction_of_profile,meta_data_mode,meta_profile_date,meta_profile_latitude,meta_profile_longitude,meta_pressure,meta_temperature,meta_salinity
0,2.0,NaN,NaN,NaN,NaN,NaN,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,4901639,0,A,D,2015-11-21T03:09:06Z,33.254,-122.475,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
1,5.0,NaN,NaN,NaN,NaN,NaN,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,4901639,0,A,D,2015-11-21T03:09:06Z,33.254,-122.475,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
2,10.0,NaN,NaN,NaN,NaN,NaN,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,4901639,0,A,D,2015-11-21T03:09:06Z,33.254,-122.475,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
3,15.0,NaN,NaN,NaN,NaN,NaN,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,4901639,0,A,D,2015-11-21T03:09:06Z,33.254,-122.475,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
4,20.0,NaN,NaN,NaN,NaN,NaN,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,4901639,0,A,D,2015-11-21T03:09:06Z,33.254,-122.475,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity


In [35]:
# Save the top-platform dataset to CSV for reuse.
single_argo.to_csv("single_argo.csv", index=False)
print("Saved single_argo.csv for platform", best_socal_platform)

Saved single_argo.csv for platform 4901639


## Southern California Coastal Platform Search

This section uses the EasyOneArgoTSLite index file to find platform numbers near the Southern California coast, ranks them by indexed profile count, and uses the top platform to build `single_argo`.

In [29]:
from pathlib import Path
import pandas as pd

index_path = Path("EasyOneArgoTSLite_20260316T043037Z/EasyOneArgoTSLite_index.csv")

# The first 5 lines are metadata comments, so skip them and read the table header.
index_df = pd.read_csv(index_path, skiprows=5)

# Southern California coastal box. Adjust if you want a narrower or wider region.
sc_mask = (
    index_df["profile_latitude"].between(32, 34.8)
    & index_df["profile_longitude"].between(-125, -117)
)
sc_index = index_df.loc[sc_mask].copy()

# Group by platform number and count how many indexed profiles each platform has in the region.
platform_counts = (
    sc_index.groupby("platform_number")
    .size()
    .sort_values(ascending=False)
)

top_10_socal_platforms = platform_counts.head(10)
best_socal_platform = top_10_socal_platforms.index[0]

print(f"Southern California coastal rows: {len(sc_index):,}")
print("Top 10 platform numbers by profile count in the region:")
print(top_10_socal_platforms.to_string())
print()
print(f"Best match: platform_number {best_socal_platform} with {top_10_socal_platforms.iloc[0]} profiles")

# Optional: inspect the platform's latitude/longitude range.
best_platform_rows = sc_index[sc_index["platform_number"] == best_socal_platform]
print()
print("Best platform latitude range:", (best_platform_rows["profile_latitude"].min(), best_platform_rows["profile_latitude"].max()))
print("Best platform longitude range:", (best_platform_rows["profile_longitude"].min(), best_platform_rows["profile_longitude"].max()))

Southern California coastal rows: 2,157
Top 10 platform numbers by profile count in the region:
platform_number
4901639    161
5904318    124
4903296    106
2903886    100
4900848     93
4901640     88
4901408     87
5906445     80
4900626     76
5906294     70

Best match: platform_number 4901639 with 161 profiles

Best platform latitude range: (np.float64(32.005), np.float64(34.606))
Best platform longitude range: (np.float64(-124.518), np.float64(-121.294))


In [36]:
# Overwrite single_argo.csv using the top Southern California platform.
single_argo = combine_csvs_from_subfolders(
    Path("EasyOneArgoTSLite_20260316T043037Z/data") / str(best_socal_platform),
    read_kwargs={"comment": "#"},
    include_metadata=True,
)

single_argo.to_csv("single_argo.csv", index=False)
print("Replaced single_argo.csv with platform", best_socal_platform)
print("Rows:", len(single_argo))

Replaced single_argo.csv with platform 4901639
Rows: 17816
